In [53]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge

In [5]:
def load_data(url):
    
    df = pd.read_csv(url)
    
    return df

In [38]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

df = load_data(url)

print(df.shape)
print(df.head())
print(df.dtypes)

(10000, 11)
   model_year  origin fuel_type         drivetrain  num_doors  \
0        2006  Europe  Gasoline  Front-wheel drive          4   
1        2008  Europe    Diesel  Front-wheel drive          4   
2        1996    Asia  Gasoline  Front-wheel drive          5   
3        1989  Europe  Gasoline  Front-wheel drive          4   
4        1994     USA    Diesel  Front-wheel drive          3   

   engine_displacement  num_cylinders  horsepower  vehicle_weight  \
0                 2180              6       243.0            3870   
1                 2390              6       272.0            4210   
2                 2320              6       267.0            4240   
3                 2130              6       258.0            4490   
4                 2580              7       304.0            4510   

   acceleration  fuel_efficiency_mpg  
0           NaN                 31.9  
1           NaN                 31.3  
2          17.3                 27.5  
3          18.7           

In [39]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)
for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')


In [40]:
df.isna().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [41]:
missing_counts = df.isna().sum()
print(missing_counts[missing_counts >0])

horsepower      877
acceleration    264
dtype: int64


In [42]:
mean_hp = df['horsepower'].mean()

In [43]:
mean_acc = df['acceleration'].mean()

In [44]:
df['horsepower'] = df['horsepower'].fillna(mean_hp)
df['acceleration'] = df['acceleration'].fillna(mean_acc)

In [45]:
df.isna().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [46]:
# Convert integer columns that represent categories into strings
categorical_columns.extend(['num_doors', 'num_cylinders'])

for c in ['num_doors', 'num_cylinders']:
    df[c] = df[c].astype(str)

In [47]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [48]:
df.dtypes

model_year               int64
origin                     str
fuel_type                  str
drivetrain                 str
num_doors                  str
engine_displacement      int64
num_cylinders              str
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

In [49]:
dv = DictVectorizer(sparse=False)

train_dict = df_train.to_dict(orient='records')


In [66]:
X_train = dv.fit_transform(train_dict)

In [67]:
dv

DictVectorizer(sparse=False)

In [68]:
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

In [69]:
val_dict = df_val.to_dict(orient='records')

In [70]:
X_val = dv.transform(val_dict)

In [71]:
dv

DictVectorizer(sparse=False)

In [72]:
y_pred = model.predict(X_val)

In [74]:
from sklearn.metrics import root_mean_squared_error
# Or: from sklearn.metrics import mean_squared_error; rmse = mean_squared_error(y_val, y_pred, squared=False)

rmse = root_mean_squared_error(y_val, y_pred)
print(f"RMSE: {rmse:.3f}")

RMSE: 1.669


In [75]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_val, y_pred)
print(f"MAE: {mae:.3f}")

MAE: 1.339


In [76]:
from sklearn.metrics import r2_score

r2 = r2_score(y_val, y_pred)
print(f"R2 Score: {r2:.3f}")

R2 Score: 0.682


In [80]:
for a in [0.0, 0.01, 0.1, 1.0, 10, 100]:
    model = Ridge(alpha=a)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)
    rmse_a = root_mean_squared_error(y_val, y_pred)
    mae_a = mean_absolute_error(y_val, y_pred)
    r2_a = r2_score(y_val, y_pred)
    
    print(f"alpha={a:5.2f}  ---> RMSE: {rmse_a:.3f}  ---> MAE: {mae_a:.3f}  ---> R square: {r2_a:.3f}")

alpha= 0.00  ---> RMSE: 1.684  ---> MAE: 1.348  ---> R square: 0.676
alpha= 0.01  ---> RMSE: 1.669  ---> MAE: 1.339  ---> R square: 0.682
alpha= 0.10  ---> RMSE: 1.669  ---> MAE: 1.339  ---> R square: 0.682
alpha= 1.00  ---> RMSE: 1.669  ---> MAE: 1.339  ---> R square: 0.682
alpha=10.00  ---> RMSE: 1.669  ---> MAE: 1.339  ---> R square: 0.682
alpha=100.00  ---> RMSE: 1.671  ---> MAE: 1.344  ---> R square: 0.681


In [83]:
for a in [0.00001, 0.0001, 0.001, 0.02, 0.05, 0.1]:
    model = Ridge(alpha=a)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)
    rmse_a = root_mean_squared_error(y_val, y_pred)
    mae_a = mean_absolute_error(y_val, y_pred)
    r2_a = r2_score(y_val, y_pred)
    
    print(f"alpha={a} ---> RMSE: {rmse_a:.3f} ---> MAE: {mae_a:.3f} ---> R square: {r2_a:.3f}")

alpha=1e-05 ---> RMSE: 1.669 ---> MAE: 1.339 ---> R square: 0.682
alpha=0.0001 ---> RMSE: 1.669 ---> MAE: 1.339 ---> R square: 0.682
alpha=0.001 ---> RMSE: 1.669 ---> MAE: 1.339 ---> R square: 0.682
alpha=0.02 ---> RMSE: 1.669 ---> MAE: 1.339 ---> R square: 0.682
alpha=0.05 ---> RMSE: 1.669 ---> MAE: 1.339 ---> R square: 0.682
alpha=0.1 ---> RMSE: 1.669 ---> MAE: 1.339 ---> R square: 0.682
